In [1]:
import polars as pl
import polars.selectors as cs
import json
from typing import Optional
import gc
import numpy as np

In [2]:
pl.Config.set_tbl_rows(-1)      
pl.Config.set_tbl_cols(-1)     
pl.Config.set_tbl_width_chars(200)  
pl.Config.set_fmt_str_lengths(50)  

polars.config.Config

In [3]:
DATA_ROOT = "D:\\Projects\\Coding\\Notebooks\\Projects\\labor_market_analysis\\data\\raw\\"
VACANCY_ROOT = DATA_ROOT + "vacancies_trudvsem.csv"

# **СБОР ФИЧЕЙ**

In [4]:
feature_columns = [
    # Keys
    'codeProfessionalSphere',
    'regionName',
    'federalDistrictCode',
    'year',
    'month',
    
    # Current Stats
    'vacancy_count',
    'median_salary',
    'avg_experience',
    'share_remote',
    'share_high_education',
    'share_accommodation',
    'share_small_company',
    
    # Time Encoding
    'month_sin',
    'month_cos',
    
    # Lag Features
    'median_salary_lag_1',
    'median_salary_lag_2',
    'median_salary_roll_mean_2',
    'median_salary_roll_std_2'
]

target_column = 'target_median_salary'

In [5]:
# ------------------------------------------------------------------------------
# Шаг 1.2: Определение колонок для загрузки
# ------------------------------------------------------------------------------
# Загружаем только те колонки, которые понадобятся для формирования признаков
columns_to_load = [
    # Ключи агрегации
    'codeProfessionalSphere',
    'regionName',
    'federalDistrictCode',
    'creationDate',
    
    # Зарплата
    'salaryMin',
    'salaryMax',
    
    # Для флагов и долей
    'busyType',
    'educationRequirements',
    'companyBusinessSize',
    'accommodationCapability',
    'experienceRequirements',
    
    # Фильтры
    'deleted',
    'status'
]

# ------------------------------------------------------------------------------
# Шаг 1.3: Определение типов данных для оптимизации памяти
# ------------------------------------------------------------------------------
schema_overrides = {
    # Числовые колонки
    'salaryMin': pl.Float32,
    'salaryMax': pl.Float32,
    'experienceRequirements': pl.Float32,
    'federalDistrictCode': pl.Int16,
    
    # Булевы колонки
    'deleted': pl.Boolean,
    'accommodationCapability': pl.Boolean,
    
    # Категориальные колонки
    'busyType': pl.Categorical,
    'educationRequirements': pl.Categorical,
    'companyBusinessSize': pl.Categorical,
    'status': pl.Categorical,
}

# ------------------------------------------------------------------------------
# Шаг 1.4: Чтение CSV с оптимизированными параметрами
# ------------------------------------------------------------------------------
print("Загрузка данных...")

df_raw = pl.read_csv(
    source=VACANCY_ROOT,
    columns=columns_to_load,
    schema_overrides=schema_overrides,
    encoding='utf-8',
    separator='|',
    try_parse_dates=True,
    ignore_errors=True,
    null_values=['', 'NULL', 'null', 'None', 'N/A', 'не указано'],
    truncate_ragged_lines=True,
    quote_char=None,
    infer_schema_length=10000,
    low_memory=True,
)

print(f"Загружено строк: {df_raw.height:,}")
print(f"Загружено колонок: {df_raw.width}")
print(f"Использование памяти: {df_raw.estimated_size('mb'):.2f} MB")

# ------------------------------------------------------------------------------
# Шаг 1.5: Базовая фильтрация данных
# ------------------------------------------------------------------------------
print("\nПрименение фильтров...")

# Фильтр 1: Только активные вакансии (не удаленные)
df_filtered = df_raw.filter(pl.col('deleted') == False)
print(f"После фильтрации deleted: {df_filtered.height:,} строк")

# Фильтр 2: Только одобренные вакансии
df_filtered = df_filtered.filter(pl.col('status') == 'Одобрено')
print(f"После фильтрации status: {df_filtered.height:,} строк")

# Фильтр 3: Корректные зарплаты (от 1000 до 1 000 000 рублей)
df_filtered = df_filtered.filter(
    (pl.col('salaryMin') >= 1000) & 
    (pl.col('salaryMin') <= 1_000_000)
)
print(f"После фильтрации salaryMin: {df_filtered.height:,} строк")

# Фильтр 4: Удаление строк с null в ключевых колонках
df_filtered = df_filtered.filter(
    pl.col('codeProfessionalSphere').is_not_null() &
    pl.col('regionName').is_not_null() &
    pl.col('creationDate').is_not_null()
)
print(f"После фильтрации null в ключевых колонках: {df_filtered.height:,} строк")

# ------------------------------------------------------------------------------
# Шаг 1.6: Проверка результата
# ------------------------------------------------------------------------------
print("\n" + "="*70)
print("ИТОГИ ЭТАПА 1")
print("="*70)
print(f"Исходное количество строк: {df_raw.height:,}")
print(f"После фильтрации: {df_filtered.height:,}")
print(f"Потеряно строк: {df_raw.height - df_filtered.height:,} ({(1 - df_filtered.height / df_raw.height) * 100:.1f}%)")
print(f"\nКолонки в датафрейме: {df_filtered.columns}")
print(f"\nТипы данных:\n{df_filtered.dtypes}")

# ------------------------------------------------------------------------------
# Шаг 1.7: Сохранение промежуточного результата (опционально)
# ------------------------------------------------------------------------------
# df_filtered.write_parquet(DATA_ROOT / "processed" / "vacancies_filtered.parquet")
# print("\nДанные сохранены в vacancies_filtered.parquet")

Загрузка данных...
Загружено строк: 452,979
Загружено колонок: 13
Использование памяти: 37.27 MB

Применение фильтров...
После фильтрации deleted: 452,971 строк
После фильтрации status: 452,506 строк
После фильтрации salaryMin: 443,025 строк
После фильтрации null в ключевых колонках: 442,526 строк

ИТОГИ ЭТАПА 1
Исходное количество строк: 452,979
После фильтрации: 442,526
Потеряно строк: 10,453 (2.3%)

Колонки в датафрейме: ['codeProfessionalSphere', 'busyType', 'educationRequirements', 'experienceRequirements', 'companyBusinessSize', 'federalDistrictCode', 'accommodationCapability', 'creationDate', 'deleted', 'regionName', 'status', 'salaryMin', 'salaryMax']

Типы данных:
[String, Categorical, Categorical, Float32, Categorical, Int16, Boolean, Datetime(time_unit='us', time_zone='UTC'), Boolean, String, Categorical, Float32, Float32]


In [6]:
df_filtered.head()

codeProfessionalSphere,busyType,educationRequirements,experienceRequirements,companyBusinessSize,federalDistrictCode,accommodationCapability,creationDate,deleted,regionName,status,salaryMin,salaryMax
str,cat,cat,f32,cat,i16,bool,"datetime[μs, UTC]",bool,str,cat,f32,f32
"""Medicine""","""Полная занятость""","""{""educationType"": ""Среднее профессиональное образо…",0.0,"""LARGE""",5,false,2025-07-24 05:32:24 UTC,false,"""Тюменская область""","""Одобрено""",42000.0,45000.0
"""Medicine""","""Полная занятость""","""{""educationType"": ""Среднее профессиональное образо…",0.0,"""LARGE""",5,false,2025-08-13 08:00:23 UTC,false,"""Тюменская область""","""Одобрено""",51300.0,56000.0
"""Medicine""","""Полная занятость""","""{""educationType"": ""Среднее профессиональное образо…",0.0,"""LARGE""",5,false,2025-06-18 10:32:07 UTC,false,"""Тюменская область""","""Одобрено""",49000.0,60000.0
"""Medicine""","""Полная занятость""","""{""educationType"": ""Высшее образование — бакалавриа…",0.0,"""LARGE""",5,false,2024-06-19 08:25:47 UTC,false,"""Тюменская область""","""Одобрено""",50470.0,53000.0
"""Medicine""","""Полная занятость""","""{""educationType"": ""Среднее профессиональное образо…",0.0,"""LARGE""",5,false,2025-01-21 05:28:26 UTC,false,"""Тюменская область""","""Одобрено""",68000.0,80000.0


In [7]:
import polars as pl
import json
from typing import Optional
from datetime import datetime

# ==============================================================================
# ЭТАП 2: Извлечение признаков (Feature Extraction)
# ==============================================================================
# Цель: Преобразовать сырые колонки в целевые признаки для агрегации
# Фильтр: Только данные с 2022 по 2026 год

# ------------------------------------------------------------------------------
# Шаг 2.1: Фильтрация по дате публикации (2022-2026)
# ------------------------------------------------------------------------------
print("Фильтрация данных по периоду 2022-2026...")

df_features = df_filtered.with_columns(
    pl.col('creationDate').dt.year().alias('creation_year')
)

df_features = df_features.filter(
    (pl.col('creation_year') >= 2022) &
    (pl.col('creation_year') <= 2026)
)

print(f"Записей после фильтрации по датам: {df_features.height:,}")
print(f"Потеряно записей: {df_filtered.height - df_features.height:,}")

# ------------------------------------------------------------------------------
# Шаг 2.2: Функция для парсинга JSON из educationRequirements
# ------------------------------------------------------------------------------
def parse_education_json(json_str: Optional[str]) -> Optional[str]:
    """
    Извлекает educationType из JSON-строки поля educationRequirements.
    Возвращает None если строка пуста, null или невалидный JSON.
    """
    if json_str is None or json_str == '' or json_str == 'null':
        return None
    
    try:
        data = json.loads(json_str.strip())
        if isinstance(data, dict):
            return data.get('educationType', None)
        return None
    except (json.JSONDecodeError, TypeError, AttributeError):
        return None

# ------------------------------------------------------------------------------
# Шаг 2.3: Функция для маппинга образования в категории
# ------------------------------------------------------------------------------
def map_education_type(edu_type: Optional[str]) -> str:
    """
    Приводит различные варианты написания образования к стандартным категориям.
    
    Категории:
    - HIGH: Высшее образование (все варианты)
    - MIDDLE_SPECIAL: Среднее профессиональное
    - MIDDLE: Среднее общее
    - NONE: Требования не предъявляются / Не указано
    """
    if edu_type is None:
        return 'NONE'
    
    edu_lower = edu_type.lower()
    
    # Высшее образование (все варианты)
    high_keywords = [
        'высшее', 'бакалавриат', 'специалитет', 'магистратура', 
        'аспирантура', 'доктор', 'подготовка кадров высшей квалификации'
    ]
    if any(keyword in edu_lower for keyword in high_keywords):
        return 'HIGH'
    
    # Среднее профессиональное
    middle_special_keywords = [
        'среднее профессиональное', 'спо', 'колледж', 'техникум'
    ]
    if any(keyword in edu_lower for keyword in middle_special_keywords):
        return 'MIDDLE_SPECIAL'
    
    # Среднее общее (школа)
    middle_keywords = [
        'среднее общее', '11 классов', '9 классов', 'общее образование'
    ]
    if any(keyword in edu_lower for keyword in middle_keywords):
        return 'MIDDLE'
    
    # Не указано / Требования не предъявляются
    none_keywords = [
        'не указано', 'требования не предъявляются', 'не имеет значения', 
        'любое', 'не требуется'
    ]
    if any(keyword in edu_lower for keyword in none_keywords):
        return 'NONE'
    
    # По умолчанию — NONE (консервативный подход)
    return 'NONE'

# ------------------------------------------------------------------------------
# Шаг 2.4: Извлечение типа образования из JSON
# ------------------------------------------------------------------------------
print("Извлечение типа образования из JSON...")

df_features = df_features.with_columns(
    pl.col('educationRequirements')
    .map_elements(parse_education_json, return_dtype=pl.String)
    .alias('education_type_raw')
)

# ------------------------------------------------------------------------------
# Шаг 2.5: Маппинг образования в стандартные категории
# ------------------------------------------------------------------------------
print("Категоризация требований к образованию...")

df_features = df_features.with_columns(
    pl.col('education_type_raw')
    .map_elements(map_education_type, return_dtype=pl.Categorical)
    .alias('education_category')
)

# ------------------------------------------------------------------------------
# Шаг 2.6: Создание флага высшего образования (для гипотезы H2)
# ------------------------------------------------------------------------------
print("Создание флага высшего образования...")

df_features = df_features.with_columns(
    (pl.col('education_category') == 'HIGH').cast(pl.Int8).alias('is_high_education')
)

# ------------------------------------------------------------------------------
# Шаг 2.7: Создание флага удалённой занятости (для гипотезы H3)
# ------------------------------------------------------------------------------
print("Создание флага удалённой занятости...")

df_features = df_features.with_columns(
    pl.when(
        pl.col('busyType').cast(pl.String).str.to_uppercase().str.contains('УДАЛЁННАЯ|REMOTE')
    )
    .then(1)
    .otherwise(0)
    .cast(pl.Int8)
    .alias('is_remote')
)

# ------------------------------------------------------------------------------
# Шаг 2.8: Создание флага малой компании (для гипотезы H1)
# ------------------------------------------------------------------------------
print("Создание флага малой компании...")

df_features = df_features.with_columns(
    pl.when(
        pl.col('companyBusinessSize').cast(pl.String).str.to_uppercase().is_in(['SMALL', 'MICRO'])
    )
    .then(1)
    .otherwise(0)
    .cast(pl.Int8)
    .alias('is_small_company')
)

# ------------------------------------------------------------------------------
# Шаг 2.9: Обработка поля предоставления жилья (для гипотезы H4)
# ------------------------------------------------------------------------------
print("Обработка флага предоставления жилья...")

df_features = df_features.with_columns(
    pl.when(pl.col('accommodationCapability') == True)
    .then(1)
    .otherwise(0)
    .cast(pl.Int8)
    .alias('has_accommodation')
)

# ------------------------------------------------------------------------------
# Шаг 2.10: Обработка опыта работы (приведение к числовому виду)
# ------------------------------------------------------------------------------
print("Обработка опыта работы...")

df_features = df_features.with_columns(
    pl.col('experienceRequirements')
    .fill_null(0)
    .clip(lower_bound=0, upper_bound=50)
    .alias('experience_years')
)

# ------------------------------------------------------------------------------
# Шаг 2.11: Расчёт медианной зарплаты (середина вилки)
# ------------------------------------------------------------------------------
print("Расчёт медианной зарплаты...")

df_features = df_features.with_columns(
    pl.when(
        (pl.col('salaryMax').is_null()) |
        (pl.col('salaryMax') == pl.col('salaryMin')) |
        (pl.col('salaryMax') <= 0)
    )
    .then(pl.col('salaryMin'))
    .otherwise((pl.col('salaryMin') + pl.col('salaryMax')) / 2)
    .alias('salary_median')
)

# ------------------------------------------------------------------------------
# Шаг 2.12: Извлечение временных признаков из creationDate
# ------------------------------------------------------------------------------
print("Извлечение временных признаков...")

# Шаг 1: Сначала создай базовые временные признаки
df_features = df_features.with_columns([
    pl.col('creationDate').dt.year().alias('year'),
    pl.col('creationDate').dt.month().alias('month'),
    pl.col('creationDate').dt.quarter().alias('quarter'),
])

# Шаг 2: ПОСЛЕ этого создай sin/cos
df_features = df_features.with_columns(
    # Циклическое кодирование месяца
    (2 * np.pi * pl.col('month') / 12).sin().alias('month_sin'),
    (2 * np.pi * pl.col('month') / 12).cos().alias('month_cos')
)


Фильтрация данных по периоду 2022-2026...
Записей после фильтрации по датам: 439,392
Потеряно записей: 3,134
Извлечение типа образования из JSON...
Категоризация требований к образованию...
Создание флага высшего образования...
Создание флага удалённой занятости...
Создание флага малой компании...
Обработка флага предоставления жилья...
Обработка опыта работы...
Расчёт медианной зарплаты...
Извлечение временных признаков...


In [8]:
df_features.head()

codeProfessionalSphere,busyType,educationRequirements,experienceRequirements,companyBusinessSize,federalDistrictCode,accommodationCapability,creationDate,deleted,regionName,status,salaryMin,salaryMax,creation_year,education_type_raw,education_category,is_high_education,is_remote,is_small_company,has_accommodation,experience_years,salary_median,year,month,quarter,month_sin,month_cos
str,cat,cat,f32,cat,i16,bool,"datetime[μs, UTC]",bool,str,cat,f32,f32,i32,str,cat,i8,i8,i8,i8,f32,f32,i32,i8,i8,f64,f64
"""Medicine""","""Полная занятость""","""{""educationType"": ""Среднее профессиональное образо…",0.0,"""LARGE""",5,false,2025-07-24 05:32:24 UTC,false,"""Тюменская область""","""Одобрено""",42000.0,45000.0,2025,"""Среднее профессиональное образование""","""MIDDLE_SPECIAL""",0,0,0,0,0.0,43500.0,2025,7,3,-0.5,-0.866025
"""Medicine""","""Полная занятость""","""{""educationType"": ""Среднее профессиональное образо…",0.0,"""LARGE""",5,false,2025-08-13 08:00:23 UTC,false,"""Тюменская область""","""Одобрено""",51300.0,56000.0,2025,"""Среднее профессиональное образование""","""MIDDLE_SPECIAL""",0,0,0,0,0.0,53650.0,2025,8,3,-0.866025,-0.5
"""Medicine""","""Полная занятость""","""{""educationType"": ""Среднее профессиональное образо…",0.0,"""LARGE""",5,false,2025-06-18 10:32:07 UTC,false,"""Тюменская область""","""Одобрено""",49000.0,60000.0,2025,"""Среднее профессиональное образование""","""MIDDLE_SPECIAL""",0,0,0,0,0.0,54500.0,2025,6,2,1.2246e-16,-1.0
"""Medicine""","""Полная занятость""","""{""educationType"": ""Высшее образование — бакалавриа…",0.0,"""LARGE""",5,false,2024-06-19 08:25:47 UTC,false,"""Тюменская область""","""Одобрено""",50470.0,53000.0,2024,"""Высшее образование — бакалавриат""","""HIGH""",1,0,0,0,0.0,51735.0,2024,6,2,1.2246e-16,-1.0
"""Medicine""","""Полная занятость""","""{""educationType"": ""Среднее профессиональное образо…",0.0,"""LARGE""",5,false,2025-01-21 05:28:26 UTC,false,"""Тюменская область""","""Одобрено""",68000.0,80000.0,2025,"""Среднее профессиональное образование""","""MIDDLE_SPECIAL""",0,0,0,0,0.0,74000.0,2025,1,1,0.5,0.866025


In [9]:
# ------------------------------------------------------------------------------
# Шаг 2.13: Проверка результата
# ------------------------------------------------------------------------------
print("="*70)
print("ИТОГИ ЭТАПА 2: ИЗВЛЕЧЕНИЕ ПРИЗНАКОВ")
print("="*70)
print(f"Исходное количество записей: {df_filtered.height:,}")
print(f"После фильтрации по датам: {df_features.height:,}")
print(f"\nКолонки в датафрейме: {df_features.columns}")
print(f"\nРаспределение по годам:")
print(df_features['year'].value_counts(sort=True))
print(f"\nРаспределение по категориям образования:")
print(df_features['education_category'].value_counts(sort=True))
print(f"\nРаспределение по типу занятости:")
print(df_features['busyType'].value_counts(sort=True))
print(f"\nРаспределение по размеру компании:")
print(df_features['companyBusinessSize'].value_counts(sort=True))
print(f"\nСтатистика по зарплате:")
print(df_features.select([
    pl.col('salary_median').min().alias('min'),
    pl.col('salary_median').max().alias('max'),
    pl.col('salary_median').median().alias('median'),
    pl.col('salary_median').mean().alias('mean')
]))

# ------------------------------------------------------------------------------
# Шаг 2.14: Сохранение промежуточного результата (опционально)
# ------------------------------------------------------------------------------
# df_features.write_parquet(DATA_ROOT / "processed" / "features_extracted.parquet")
# print("\nДанные сохранены в features_extracted.parquet")


ИТОГИ ЭТАПА 2: ИЗВЛЕЧЕНИЕ ПРИЗНАКОВ
Исходное количество записей: 442,526
После фильтрации по датам: 439,392

Колонки в датафрейме: ['codeProfessionalSphere', 'busyType', 'educationRequirements', 'experienceRequirements', 'companyBusinessSize', 'federalDistrictCode', 'accommodationCapability', 'creationDate', 'deleted', 'regionName', 'status', 'salaryMin', 'salaryMax', 'creation_year', 'education_type_raw', 'education_category', 'is_high_education', 'is_remote', 'is_small_company', 'has_accommodation', 'experience_years', 'salary_median', 'year', 'month', 'quarter', 'month_sin', 'month_cos']

Распределение по годам:
shape: (5, 2)
┌──────┬────────┐
│ year ┆ count  │
│ ---  ┆ ---    │
│ i32  ┆ u32    │
╞══════╪════════╡
│ 2026 ┆ 274042 │
│ 2025 ┆ 123596 │
│ 2024 ┆ 19332  │
│ 2023 ┆ 11917  │
│ 2022 ┆ 10505  │
└──────┴────────┘

Распределение по категориям образования:
shape: (4, 2)
┌────────────────────┬────────┐
│ education_category ┆ count  │
│ ---                ┆ ---    │
│ cat        

In [10]:
import polars as pl
from datetime import datetime

# ==============================================================================
# ЭТАП 3: Агрегация (GroupBy)
# ==============================================================================
# Цель: Перейти от уровня вакансии к уровню прогноза
# (Сфера × Регион × Месяц)

# ------------------------------------------------------------------------------
# Шаг 3.1: Проверка входных данных
# ------------------------------------------------------------------------------
print("Начало этапа агрегации...")
print(f"Количество записей перед агрегацией: {df_features.height:,}")

# Проверка ключевых колонок
required_columns = [
    'year', 'month', 'codeProfessionalSphere', 'regionName',
    'federalDistrictCode', 'salary_median', 'experience_years',
    'is_remote', 'education_category', 'accommodationCapability',
    'companyBusinessSize'
]

missing_columns = [col for col in required_columns if col not in df_features.columns]
if missing_columns:
    raise ValueError(f"Отсутствуют необходимые колонки: {missing_columns}")

# ------------------------------------------------------------------------------
# Шаг 3.2: Создание флага высшего образования (бинарный)
# ------------------------------------------------------------------------------
print("Создание бинарных признаков для агрегации...")

df_features = df_features.with_columns(
    # Флаг высшего образования (1 если HIGH, иначе 0)
    (pl.col('education_category') == 'HIGH').cast(pl.Int8).alias('is_high_education')
)

# ------------------------------------------------------------------------------
# Шаг 3.3: Создание флага малой компании (бинарный)
# ------------------------------------------------------------------------------
df_features = df_features.with_columns(
    # Флаг малой компании (MICRO или SMALL)
    (pl.col('companyBusinessSize').cast(pl.String).str.to_uppercase().is_in(['SMALL', 'MICRO']))
    .cast(pl.Int8)
    .alias('is_small_company')
)

# ------------------------------------------------------------------------------
# Шаг 3.4: Фильтрация записей с null в ключевых полях группировки
# ------------------------------------------------------------------------------
print("Фильтрация записей с null в ключевых полях...")

df_features = df_features.filter(
    pl.col('year').is_not_null() &
    pl.col('month').is_not_null() &
    pl.col('codeProfessionalSphere').is_not_null() &
    pl.col('regionName').is_not_null() &
    pl.col('salary_median').is_not_null()
)

print(f"Количество записей после фильтрации: {df_features.height:,}")
print(f"Потеряно записей: {df_features.height - df_features.height:,}")

# ------------------------------------------------------------------------------
# Шаг 3.5: Агрегация по ключам (Сфера × Регион × Месяц)
# ------------------------------------------------------------------------------
print("Выполнение агрегации...")

df_aggregated = (
    df_features
    .group_by('year', 'month', 'codeProfessionalSphere', 'regionName', 'federalDistrictCode')
    .agg(
        # Количество вакансий в группе
        pl.len().alias('vacancy_count'),
        
        # Медианная зарплата
        pl.col('salary_median').median().alias('median_salary'),
        
        # Средний требуемый опыт работы
        pl.col('experience_years').mean().alias('avg_experience'),
        
        # Доля удаленных вакансий (среднее от бинарного флага)
        pl.col('is_remote').mean().alias('share_remote'),
        
        # Доля вакансий с требованием высшего образования
        pl.col('is_high_education').mean().alias('share_high_education'),
        
        # Доля вакансий с предоставлением жилья
        pl.col('accommodationCapability').cast(pl.Int8).mean().alias('share_accommodation'),
        
        # Доля вакансий от малых компаний
        pl.col('is_small_company').mean().alias('share_small_company'),
        
        # Дополнительно: стандартное отклонение зарплаты (для анализа волатильности)
        pl.col('salary_median').std().alias('salary_std'),
        
        # Минимальная и максимальная зарплата в группе
        pl.col('salary_median').min().alias('salary_min'),
        pl.col('salary_median').max().alias('salary_max'),
        
        # Количество уникальных компаний в группе (аппроксимация)
        pl.col('companyBusinessSize').n_unique().alias('unique_company_sizes')
    )
)

print(f"Количество агрегированных записей: {df_aggregated.height:,}")

# ------------------------------------------------------------------------------
# Шаг 3.6: Фильтрация групп с малым количеством вакансий
# ------------------------------------------------------------------------------
print("Фильтрация групп с малым количеством вакансий...")

# Минимальное количество вакансий в группе для устойчивости статистики
MIN_VACANCIES = 4

df_aggregated = df_aggregated.filter(pl.col('vacancy_count') >= MIN_VACANCIES)

print(f"Количество записей после фильтрации по min_vacancies={MIN_VACANCIES}: {df_aggregated.height:,}")

# ------------------------------------------------------------------------------
# Шаг 3.7: Сортировка по времени и ключам
# ------------------------------------------------------------------------------
print("Сортировка данных...")

df_aggregated = df_aggregated.sort(
    ['codeProfessionalSphere', 'regionName', 'year', 'month'],
    descending=[False, False, False, False]
)

# ------------------------------------------------------------------------------
# Шаг 3.8: Проверка результата
# ------------------------------------------------------------------------------
print("\n" + "="*70)
print("ИТОГИ ЭТАПА 3: АГРЕГАЦИЯ")
print("="*70)

print(f"\nИсходное количество вакансий: {df_features.height:,}")
print(f"Количество агрегированных групп: {df_aggregated.height:,}")
print(f"Среднее количество вакансий в группе: {df_features.height / df_aggregated.height:.1f}")

print(f"\nУникальных профессиональных сфер: {df_aggregated['codeProfessionalSphere'].n_unique()}")
print(f"Уникальных регионов: {df_aggregated['regionName'].n_unique()}")
print(f"Период данных: {df_aggregated['year'].min()} - {df_aggregated['year'].max()}")
print(f"Месяцы: {df_aggregated['month'].min()} - {df_aggregated['month'].max()}")

print(f"\nСтатистика по количеству вакансий в группе:")
vacancy_stats = df_aggregated.select([
    pl.col('vacancy_count').min().alias('min'),
    pl.col('vacancy_count').max().alias('max'),
    pl.col('vacancy_count').mean().alias('mean'),
    pl.col('vacancy_count').median().alias('median')
])
print(vacancy_stats)

print(f"\nСтатистика по медианной зарплате:")
salary_stats = df_aggregated.select([
    pl.col('median_salary').min().alias('min'),
    pl.col('median_salary').max().alias('max'),
    pl.col('median_salary').mean().alias('mean'),
    pl.col('median_salary').median().alias('median')
])
print(salary_stats)

print(f"\nКолонки в агрегированном датафрейме: {df_aggregated.columns}")

# ------------------------------------------------------------------------------
# Шаг 3.11: Сохранение промежуточного результата (опционально)
# ------------------------------------------------------------------------------
# df_aggregated.write_parquet(DATA_ROOT / "processed" / "vacancies_aggregated.parquet")
# print("\nДанные сохранены в vacancies_aggregated.parquet")

print("\n" + "="*70)
print("ЭТАП 3 ЗАВЕРШЕН УСПЕШНО")
print("="*70)

Начало этапа агрегации...
Количество записей перед агрегацией: 439,392
Создание бинарных признаков для агрегации...
Фильтрация записей с null в ключевых полях...
Количество записей после фильтрации: 439,392
Потеряно записей: 0
Выполнение агрегации...
Количество агрегированных записей: 42,182
Фильтрация групп с малым количеством вакансий...
Количество записей после фильтрации по min_vacancies=4: 16,308
Сортировка данных...

ИТОГИ ЭТАПА 3: АГРЕГАЦИЯ

Исходное количество вакансий: 439,392
Количество агрегированных групп: 16,308
Среднее количество вакансий в группе: 26.9

Уникальных профессиональных сфер: 35
Уникальных регионов: 91
Период данных: 2022 - 2026
Месяцы: 1 - 12

Статистика по количеству вакансий в группе:
shape: (1, 4)
┌─────┬──────┬───────────┬────────┐
│ min ┆ max  ┆ mean      ┆ median │
│ --- ┆ ---  ┆ ---       ┆ ---    │
│ u32 ┆ u32  ┆ f64       ┆ f64    │
╞═════╪══════╪═══════════╪════════╡
│ 4   ┆ 1341 ┆ 24.508156 ┆ 10.0   │
└─────┴──────┴───────────┴────────┘

Статистика

In [11]:
df_aggregated.head()

year,month,codeProfessionalSphere,regionName,federalDistrictCode,vacancy_count,median_salary,avg_experience,share_remote,share_high_education,share_accommodation,share_small_company,salary_std,salary_min,salary_max,unique_company_sizes
i32,i8,str,str,i16,u32,f32,f32,f64,f64,f64,f64,f32,f32,f32,u32
2025,7,"""AccountingTaxesManagement""","""Алтайский край""",6,4,45000.0,3.0,0.0,1.0,0.0,1.0,4750.0,35500.0,45000.0,1
2025,12,"""AccountingTaxesManagement""","""Алтайский край""",6,9,45000.0,1.222222,0.0,0.555556,0.0,1.0,10627.150391,31157.0,62500.0,1
2026,1,"""AccountingTaxesManagement""","""Алтайский край""",6,21,35500.0,2.095238,0.0,0.333333,0.0,0.809524,8027.386719,31157.0,59500.0,4
2026,2,"""AccountingTaxesManagement""","""Алтайский край""",6,39,44500.0,1.205128,0.025641,0.333333,0.025641,0.846154,10294.139648,31157.0,70000.0,5
2026,3,"""AccountingTaxesManagement""","""Алтайский край""",6,17,41000.0,1.294118,0.0,0.470588,0.0,0.705882,15300.788086,31157.0,90000.0,3


In [12]:
import polars as pl
from datetime import datetime

# ==============================================================================
# ЭТАП 4: Создание временных рядов и лагов (Time Series Features)
# ==============================================================================
# Цель: Добавить историю (лаги) и скользящие статистики для обучения модели
#       прогнозированию медианной зарплаты на 1 месяц вперед

df_aggregated = df_aggregated.with_columns(
    (2 * np.pi * pl.col('month') / 12).sin().alias('month_sin'),
    (2 * np.pi * pl.col('month') / 12).cos().alias('month_cos')
)

# ------------------------------------------------------------------------------
# Шаг 4.1: Проверка входных данных
# ------------------------------------------------------------------------------
print("Начало этапа создания временных рядов и лагов...")
print(f"Количество записей перед обработкой: {df_aggregated.height:,}")

# Проверка обязательных колонок
required_columns = [
    'year', 'month', 'codeProfessionalSphere', 'regionName',
    'median_salary', 'vacancy_count'
]

missing_columns = [col for col in required_columns if col not in df_aggregated.columns]
if missing_columns:
    raise ValueError(f"Отсутствуют необходимые колонки: {missing_columns}")

# ------------------------------------------------------------------------------
# Шаг 4.2: Сортировка данных по времени внутри каждой группы
# ------------------------------------------------------------------------------
print("Сортировка данных по времени и группам...")

df_sorted = df_aggregated.sort(
    ['codeProfessionalSphere', 'regionName', 'year', 'month'],
    descending=[False, False, False, False]
)

# ------------------------------------------------------------------------------
# Шаг 4.3: Создание временной метки для оконных операций
# ------------------------------------------------------------------------------
print("Создание временной метки для оконных операций...")

df_sorted = df_sorted.with_columns(
    pl.struct(['year', 'month']).alias('time_key')
)

# ------------------------------------------------------------------------------
# Шаг 4.4: Создание лагов медианной зарплаты (lag_1, lag_2)
# ------------------------------------------------------------------------------
print("Расчет лагов медианной зарплаты...")

df_with_lags = df_sorted.with_columns(
    pl.col('median_salary').shift(1).over(['codeProfessionalSphere', 'regionName']).alias('median_salary_lag_1'),
    pl.col('median_salary').shift(2).over(['codeProfessionalSphere', 'regionName']).alias('median_salary_lag_2'),
    # Убрать lag_3
)

# ------------------------------------------------------------------------------
# Шаг 4.5: Расчет скользящих статистик (окно 3 месяца)
# ------------------------------------------------------------------------------
print("Расчет скользящих статистик (окно 3 месяца)...")

df_with_rolling = df_with_lags.with_columns(
    # Скользящее среднее за 2 месяца (включая текущий)
    pl.col('median_salary')
    .rolling_mean(window_size=2)
    .over(['codeProfessionalSphere', 'regionName'])
    .alias('median_salary_roll_mean_2'),
    
    # Скользящее стандартное отклонение за 2 месяца
    pl.col('median_salary')
    .rolling_std(window_size=2)
    .over(['codeProfessionalSphere', 'regionName'])
    .alias('median_salary_roll_std_2')
)

# ------------------------------------------------------------------------------
# Шаг 4.6: Создание целевой переменной (target_median_salary)
# ------------------------------------------------------------------------------
print("Создание целевой переменной (зарплата следующего месяца)...")

df_with_target = df_with_rolling.with_columns(
    # Целевая переменная: медианная зарплата следующего месяца (lead на 1 период)
    pl.col('median_salary')
    .shift(-1)
    .over(['codeProfessionalSphere', 'regionName'])
    .alias('target_median_salary')
)

# ------------------------------------------------------------------------------
# Шаг 4.7: Фильтрация строк с null значениями
# ------------------------------------------------------------------------------
print("Фильтрация строк с null значениями в лагах и таргете...")

# Сохраняем количество записей до фильтрации
before_filter_count = df_with_target.height

# Удаляем строки где есть null в лагах (первые 3 месяца истории)
# или null в таргете (последний месяц)
df_features = df_with_target.filter(
    pl.col('median_salary_lag_1').is_not_null() &
    pl.col('median_salary_lag_2').is_not_null() &
    pl.col('target_median_salary').is_not_null()
)

# Количество записей после фильтрации
after_filter_count = df_features.height
removed_count = before_filter_count - after_filter_count

print(f"Записей до фильтрации: {before_filter_count:,}")
print(f"Записей после фильтрации: {after_filter_count:,}")
print(f"Удалено записей: {removed_count:,} ({removed_count / before_filter_count * 100:.1f}%)")

# ------------------------------------------------------------------------------
# Шаг 4.8: Выбор финальных колонок для features_df
# ------------------------------------------------------------------------------
print("Выбор финальных колонок для features_df...")

feature_columns = [
    'codeProfessionalSphere',
    'regionName',
    'year',
    'month',
    
    'vacancy_count',
    'median_salary',
    
    'median_salary_lag_1',
    'median_salary_lag_2',
    'median_salary_roll_mean_2',
    'median_salary_roll_std_2',
    
    'target_median_salary'
]

additional_columns = [
    'avg_experience',
    'share_remote',
    'share_high_education',
    'share_accommodation',
    'share_small_company',
    'federalDistrictCode',
    'month_sin',    
    'month_cos'      
]

for col in additional_columns:
    if col in df_features.columns:
        feature_columns.append(col)

# Выбираем только нужные колонки
df_features = df_features.select(feature_columns)

# ------------------------------------------------------------------------------
# Шаг 4.9: Проверка результата
# ------------------------------------------------------------------------------
print("\n" + "="*70)
print("ИТОГИ ЭТАПА 4: ВРЕМЕННЫЕ РЯДЫ И ЛАГИ")
print("="*70)

print(f"\nИсходное количество записей: {df_aggregated.height:,}")
print(f"Финальное количество записей: {df_features.height:,}")
print(f"Потеряно записей: {df_aggregated.height - df_features.height:,}")
print(f"Потеряно из-за лагов/таргета: {removed_count:,}")

print(f"\nКоличество колонок: {df_features.width}")
print(f"Колонки в features_df: {df_features.columns}")

print(f"\nУникальных профессиональных сфер: {df_features['codeProfessionalSphere'].n_unique()}")
print(f"Уникальных регионов: {df_features['regionName'].n_unique()}")
print(f"Период данных: {df_features['year'].min()} - {df_features['year'].max()}")
print(f"Месяцы: {df_features['month'].min()} - {df_features['month'].max()}")

print(f"\nСтатистика по целевой переменной (target_median_salary):")
target_stats = df_features.select([
    pl.col('target_median_salary').min().alias('min'),
    pl.col('target_median_salary').max().alias('max'),
    pl.col('target_median_salary').median().alias('median'),
    pl.col('target_median_salary').mean().alias('mean'),
    pl.col('target_median_salary').std().alias('std')
])
print(target_stats)

print(f"\nСтатистика по лагам:")
lag_stats = df_features.select([
    pl.col('median_salary_lag_1').mean().alias('lag_1_mean'),
    pl.col('median_salary_lag_2').mean().alias('lag_2_mean'),
])
print(lag_stats)

# ------------------------------------------------------------------------------
# Шаг 4.10: Проверка на наличие дубликатов
# ------------------------------------------------------------------------------
print(f"\nПроверка на дубликаты...")

duplicate_check = df_features.select(
    pl.col('codeProfessionalSphere', 'regionName', 'year', 'month')
)

duplicate_count = duplicate_check.height - duplicate_check.unique().height

if duplicate_count == 0:
    print("✅ Дубликатов не обнаружено")
else:
    print(f"⚠️ Обнаружено дубликатов: {duplicate_count}")

# ------------------------------------------------------------------------------
# Шаг 4.11: Проверка корреляций между признаками
# ------------------------------------------------------------------------------
print(f"\nКорреляция лагов с целевой переменной:")

correlation_check = df_features.select([
    pl.corr('median_salary_lag_1', 'target_median_salary').alias('lag_1_corr'),
    pl.corr('median_salary_lag_2', 'target_median_salary').alias('lag_2_corr'),
    pl.corr('median_salary', 'target_median_salary').alias('current_corr')
])

print(correlation_check)

# ------------------------------------------------------------------------------
# Шаг 4.12: Сохранение промежуточного результата
# ------------------------------------------------------------------------------
# df_features.write_parquet(DATA_ROOT / "processed" / "features_with_lags.parquet")
# print("\nДанные сохранены в features_with_lags.parquet")

print("\n" + "="*70)
print("ЭТАП 4 ЗАВЕРШЕН УСПЕШНО")
print("="*70)

Начало этапа создания временных рядов и лагов...
Количество записей перед обработкой: 16,308
Сортировка данных по времени и группам...
Создание временной метки для оконных операций...
Расчет лагов медианной зарплаты...
Расчет скользящих статистик (окно 3 месяца)...
Создание целевой переменной (зарплата следующего месяца)...
Фильтрация строк с null значениями в лагах и таргете...
Записей до фильтрации: 16,308
Записей после фильтрации: 9,914
Удалено записей: 6,394 (39.2%)
Выбор финальных колонок для features_df...

ИТОГИ ЭТАПА 4: ВРЕМЕННЫЕ РЯДЫ И ЛАГИ

Исходное количество записей: 16,308
Финальное количество записей: 9,914
Потеряно записей: 6,394
Потеряно из-за лагов/таргета: 6,394

Количество колонок: 19
Колонки в features_df: ['codeProfessionalSphere', 'regionName', 'year', 'month', 'vacancy_count', 'median_salary', 'median_salary_lag_1', 'median_salary_lag_2', 'median_salary_roll_mean_2', 'median_salary_roll_std_2', 'target_median_salary', 'avg_experience', 'share_remote', 'share_high

In [13]:
df_features.head(10)

codeProfessionalSphere,regionName,year,month,vacancy_count,median_salary,median_salary_lag_1,median_salary_lag_2,median_salary_roll_mean_2,median_salary_roll_std_2,target_median_salary,avg_experience,share_remote,share_high_education,share_accommodation,share_small_company,federalDistrictCode,month_sin,month_cos
str,str,i32,i8,u32,f32,f32,f32,f32,f32,f32,f32,f64,f64,f64,f64,i16,f64,f64
"""AccountingTaxesManagement""","""Алтайский край""",2026,1,21,35500.0,45000.0,45000.0,40250.0,6717.514648,44500.0,2.095238,0.0,0.333333,0.0,0.809524,6,0.5,0.866025
"""AccountingTaxesManagement""","""Алтайский край""",2026,2,39,44500.0,35500.0,45000.0,40000.0,6363.960938,41000.0,1.205128,0.025641,0.333333,0.025641,0.846154,6,0.866025,0.5
"""AccountingTaxesManagement""","""Амурская область""",2025,10,4,54250.0,61500.0,57100.0,57875.0,5126.523926,60000.0,2.0,0.0,0.5,0.0,0.75,7,-0.866025,0.5
"""AccountingTaxesManagement""","""Амурская область""",2025,12,7,60000.0,54250.0,61500.0,57125.0,4065.864014,67500.0,2.0,0.0,0.714286,0.0,1.0,7,-2.4493e-16,1.0
"""AccountingTaxesManagement""","""Амурская область""",2026,1,13,67500.0,60000.0,54250.0,63750.0,5303.300781,52500.0,1.076923,0.0,0.769231,0.0,0.615385,7,0.5,0.866025
"""AccountingTaxesManagement""","""Амурская область""",2026,2,11,52500.0,67500.0,60000.0,60000.0,10606.601562,65000.0,0.818182,0.0,0.545455,0.0,0.818182,7,0.866025,0.5
"""AccountingTaxesManagement""","""Брянская область""",2026,2,9,30000.0,31085.75,46132.75,30542.875,767.74115,38250.0,1.222222,0.0,0.333333,0.0,1.0,1,0.866025,0.5
"""AccountingTaxesManagement""","""Владимирская область""",2026,2,18,49250.0,41033.25,46750.0,45141.625,5810.119629,48000.0,0.944444,0.0,0.444444,0.0,0.944444,1,0.866025,0.5
"""AccountingTaxesManagement""","""Волгоградская область""",2026,1,34,28773.25,53014.25,41250.0,40893.75,17140.976562,37500.0,1.941176,0.0,0.5,0.0,0.941176,3,0.5,0.866025


In [14]:
df_features.tail(10)

codeProfessionalSphere,regionName,year,month,vacancy_count,median_salary,median_salary_lag_1,median_salary_lag_2,median_salary_roll_mean_2,median_salary_roll_std_2,target_median_salary,avg_experience,share_remote,share_high_education,share_accommodation,share_small_company,federalDistrictCode,month_sin,month_cos
str,str,i32,i8,u32,f32,f32,f32,f32,f32,f32,f32,f64,f64,f64,f64,i16,f64,f64
"""WorkingSpecialties""","""Челябинская область""",2025,9,4,46575.0,52500.0,56000.0,49537.5,4189.607422,46150.0,1.5,0.0,0.25,0.0,0.75,5,-1.0,-1.8370e-16
"""WorkingSpecialties""","""Челябинская область""",2025,10,8,46150.0,46575.0,52500.0,46362.5,300.520386,42500.0,1.0,0.0,0.0,0.0,0.875,5,-0.866025,0.5
"""WorkingSpecialties""","""Челябинская область""",2025,11,4,42500.0,46150.0,46575.0,44325.0,2580.939697,40800.0,0.5,0.0,0.0,0.0,0.0,5,-0.5,0.866025
"""WorkingSpecialties""","""Челябинская область""",2025,12,20,40800.0,42500.0,46150.0,41650.0,1202.081543,35000.0,0.6,0.0,0.05,0.0,0.45,5,-2.4493e-16,1.0
"""WorkingSpecialties""","""Челябинская область""",2026,1,68,35000.0,40800.0,42500.0,37900.0,4101.219238,31178.5,0.720588,0.0,0.0,0.029412,0.75,5,0.5,0.866025
"""WorkingSpecialties""","""Челябинская область""",2026,2,54,31178.5,35000.0,40800.0,33089.25,2702.208496,35221.0,0.981481,0.0,0.018519,0.0,0.407407,5,0.866025,0.5
"""WorkingSpecialties""","""Чувашская Республика - Чувашия""",2026,1,15,32195.0,59400.0,45200.0,45797.5,19236.839844,28775.0,0.2,0.0,0.066667,0.066667,0.466667,4,0.5,0.866025
"""WorkingSpecialties""","""Чувашская Республика - Чувашия""",2026,2,14,28775.0,32195.0,59400.0,30485.0,2418.305176,30046.5,0.0,0.0,0.0,0.0,1.0,4,0.866025,0.5
"""WorkingSpecialties""","""Ямало-Ненецкий автономный округ""",2026,1,19,70442.0,30786.5,47705.5,50614.25,28040.673828,67733.0,1.421053,0.0,0.157895,0.0,0.684211,5,0.5,0.866025


In [15]:
# После Этапа 4 добавить:
print("="*70)
print("ВАЛИДАЦИЯ ФИНАЛЬНОГО ДАТАФРЕЙМА")
print("="*70)

# Проверка на дубликаты
duplicates = df_features.select(
    pl.col('codeProfessionalSphere', 'regionName', 'year', 'month')
).unique().height

print(f"Уникальных комбинаций: {duplicates}")
print(f"Всего строк: {df_features.height}")
print(f"Дубликатов: {df_features.height - duplicates}")

# Проверка на NaN
null_summary = df_features.select([
    pl.col(col).null_count().alias(col) for col in df_features.columns
]).transpose(include_header=True, header_name="column")

print("\nКолонки с пропусками:")
print(null_summary.filter(pl.col("column_0") > 0))

ВАЛИДАЦИЯ ФИНАЛЬНОГО ДАТАФРЕЙМА
Уникальных комбинаций: 9914
Всего строк: 9914
Дубликатов: 0

Колонки с пропусками:
shape: (0, 2)
┌────────┬──────────┐
│ column ┆ column_0 │
│ ---    ┆ ---      │
│ str    ┆ u32      │
╞════════╪══════════╡
└────────┴──────────┘


In [16]:
# Сохранить финальный features_df
df_features.write_parquet(
    DATA_ROOT + "processed/features.parquet",
    compression='zstd'
)

In [17]:
# -----------------------------------------------------------------------------
# 7.2. Обработка пропусков в категориальных колонках
# -----------------------------------------------------------------------------

# Заполняем пропуски в categorical колонках значением 'UNKNOWN'
categorical_cols_to_fill = [
    'source_type_group',
    'busy_type_group',
    'schedule_type_group',
    'education_category',
    'premium_type',
    'companyBusinessSize'
]

for col in categorical_cols_to_fill:
    if col in feature_df.columns:
        # Проверяем есть ли пропуски
        null_count = feature_df[col].null_count()
        if null_count > 0:
            feature_df = feature_df.with_columns(
                pl.col(col).fill_null('UNKNOWN')
            )
            print(f"Заполнено {null_count} пропусков в колонке {col}")

# Заполняем пропуски в company_code
if 'company_code' in feature_df.columns:
    null_count = feature_df['company_code'].null_count()
    if null_count > 0:
        feature_df = feature_df.with_columns(
            pl.col('company_code').fill_null('UNKNOWN')
        )
        print(f"Заполнено {null_count} пропусков в колонке company_code")

NameError: name 'feature_df' is not defined

In [ ]:
# -----------------------------------------------------------------------------
# 7.3. Обработка пропусков в числовых колонках
# -----------------------------------------------------------------------------

# Заполняем пропуски в experienceRequirements медианой по сфере
if 'experienceRequirements' in feature_df.columns and 'professionalSphereName' in feature_df.columns:
    null_count = feature_df['experienceRequirements'].null_count()
    if null_count > 0:
        # Вычисляем медиану по professionalSphereName
        feature_df = feature_df.with_columns(
            pl.col('experienceRequirements')
            .fill_null(
                pl.col('experienceRequirements')
                .median().over('professionalSphereName')
            )
        )
        # Если остались пропуски (сферы без данных), заполняем общей медианой
        feature_df = feature_df.with_columns(
            pl.col('experienceRequirements')
            .fill_null(feature_df['experienceRequirements'].median())
        )
        print(f"Заполнено {null_count} пропусков в колонке experienceRequirements")

# Заполняем пропуски в workPlaces нулем
if 'workPlaces' in feature_df.columns:
    null_count = feature_df['workPlaces'].null_count()
    if null_count > 0:
        feature_df = feature_df.with_columns(
            pl.col('workPlaces').fill_null(0)
        )
        print(f"Заполнено {null_count} пропусков в колонке workPlaces")

# Заполняем пропуски в premium_size нулем
if 'premium_size' in feature_df.columns:
    null_count = feature_df['premium_size'].null_count()
    if null_count > 0:
        feature_df = feature_df.with_columns(
            pl.col('premium_size').fill_null(0)
        )
        print(f"Заполнено {null_count} пропусков в колонке premium_size")

In [ ]:
# -----------------------------------------------------------------------------
# 7.4. Обработка пропусков в булевых колонках
# -----------------------------------------------------------------------------

boolean_cols_to_fill = [
    'is_remote',
    'has_premium',
    'accommodationCapability',
    'careerPerspective',
    'is_month_end',
    'is_quarter_end',
    'is_seasonal_period',
    'is_weekend_publication'
]

for col in boolean_cols_to_fill:
    if col in feature_df.columns:
        null_count = feature_df[col].null_count()
        if null_count > 0:
            feature_df = feature_df.with_columns(
                pl.col(col).fill_null(False)
            )
            print(f"Заполнено {null_count} пропусков в колонке {col}")

In [ ]:
# =============================================================================
# ЗАПОЛНЕНИЕ ПРОПУСКОВ В regionName
# =============================================================================

# Проверяем количество пропусков перед заполнением
null_count_before = feature_df['regionName'].null_count()
print(f"Пропусков в regionName до заполнения: {null_count_before}")

if null_count_before > 0:
    # Заполняем пропуски значением 'UNKNOWN'
    # Это безопасный вариант, который не искажает распределение существующих регионов
    feature_df = feature_df.with_columns(
        pl.col('regionName').fill_null('UNKNOWN')
    )
    
    # Если вы планируете использовать эту колонку как категориальную в моделях (CatBoost/LightGBM),
    # полезно сразу привести её к типу Categorical, если это еще не сделано
    if feature_df['regionName'].dtype != pl.Categorical:
        feature_df = feature_df.with_columns(
            pl.col('regionName').cast(pl.Categorical)
        )

    # Проверяем результат
    null_count_after = feature_df['regionName'].null_count()
    print(f"Пропусков в regionName после заполнения: {null_count_after}")
    
    # Проверка уникальных значений (должно появиться 'UNKNOWN')
    print("\nУникальные значения (топ-10):")
    print(feature_df['regionName'].value_counts(sort=True).head(10))
else:
    print("Пропусков не найдено, действие не требуется.")

In [ ]:
# -----------------------------------------------------------------------------
# 7.5. Конвертация строковых колонок в категориальные
# -----------------------------------------------------------------------------

# Список колонок для конвертации в Categorical
string_to_categorical = [
    'regionName',
    'professionalSphereName',
    'company_code',
    'source_type_group',
    'busy_type_group',
    'schedule_type_group',
    'education_category',
    'premium_type',
    'companyBusinessSize'
]

for col in string_to_categorical:
    if col in feature_df.columns:
        current_dtype = feature_df[col].dtype
        if current_dtype == pl.String:
            feature_df = feature_df.with_columns(
                pl.col(col).cast(pl.Categorical)
            )
            print(f"Конвертирована колонка {col}: String → Categorical")
        elif current_dtype == pl.Categorical:
            print(f"Колонка {col} уже имеет тип Categorical")

In [ ]:
# -----------------------------------------------------------------------------
# 7.6. Оптимизация числовых типов данных
# -----------------------------------------------------------------------------

# Оптимизация типов для экономии памяти
feature_df = feature_df.with_columns([
    # Годы не превысят 32767
    pl.col('publish_year').cast(pl.Int16) if 'publish_year' in feature_df.columns else pl.lit(0),
    
    # Месяцы от базовой даты
    pl.col('month_since_base').cast(pl.Int16) if 'month_since_base' in feature_df.columns else pl.lit(0),
    
    # Дни возраста вакансии
    pl.col('vacancy_age_days').cast(pl.Int32) if 'vacancy_age_days' in feature_df.columns else pl.lit(0),
    
    # Количество рабочих мест
    pl.col('workPlaces').cast(pl.Int16) if 'workPlaces' in feature_df.columns else pl.lit(0),
    
    # Размер премии
    pl.col('premium_size').cast(pl.Int32) if 'premium_size' in feature_df.columns else pl.lit(0),
    
    # День года (1-365)
    pl.col('day_of_year').cast(pl.Int16) if 'day_of_year' in feature_df.columns else pl.lit(0),
    
    # Месяц, квартал, день недели, день месяца, неделя
    pl.col('publish_month').cast(pl.Int8) if 'publish_month' in feature_df.columns else pl.lit(0),
    pl.col('publish_quarter').cast(pl.Int8) if 'publish_quarter' in feature_df.columns else pl.lit(0),
    pl.col('publish_day_of_week').cast(pl.Int8) if 'publish_day_of_week' in feature_df.columns else pl.lit(0),
    pl.col('publish_day_of_month').cast(pl.Int8) if 'publish_day_of_month' in feature_df.columns else pl.lit(0),
    pl.col('publish_week').cast(pl.Int8) if 'publish_week' in feature_df.columns else pl.lit(0),
])

print("Оптимизированы числовые типы данных")

In [ ]:
# -----------------------------------------------------------------------------
# 7.7. Удаление технических колонок
# -----------------------------------------------------------------------------

# Колонки для удаления (технические, не нужны для модели)
columns_to_drop = [
    'id',              # Технический идентификатор
    'salaryMin',       # Используется только для создания target
    'salaryMax',       # Используется только для создания target
    'salary',          # Дублирующая информация
]

# Фильтруем только существующие колонки
existing_cols_to_drop = [col for col in columns_to_drop if col in feature_df.columns]

if existing_cols_to_drop:
    feature_df = feature_df.drop(existing_cols_to_drop)
    print(f"Удалены технические колонки: {existing_cols_to_drop}")
else:
    print("Технические колонки уже удалены или отсутствуют")

In [ ]:
# -----------------------------------------------------------------------------
# 7.8. Финальная валидация датафрейма
# -----------------------------------------------------------------------------

print("=" * 70)
print("ФИНАЛЬНАЯ ПРОВЕРКА ПОСЛЕ ЭТАПА 7")
print("=" * 70)
print(f"Записей в feature_df: {feature_df.height:,}")
print(f"Количество колонок: {feature_df.width}")

# Проверка пропусков после оптимизации
null_summary_final = feature_df.select([
    pl.col(col).null_count().alias(col) for col in feature_df.columns
]).transpose(include_header=True, header_name="column")

null_summary_final = null_summary_final.rename({"column_0": "null_count"})
null_summary_final = null_summary_final.with_columns(
    ((pl.col("null_count") / feature_df.height) * 100).round(2).alias("null_percentage")
)
null_summary_final = null_summary_final.sort("null_percentage", descending=True)
null_summary_final_filtered = null_summary_final.filter(pl.col("null_percentage") > 0)

print(f"\nКолонок с пропусками после оптимизации: {null_summary_final_filtered.height}")
if null_summary_final_filtered.height > 0:
    print("\nКолонки с оставшимися пропусками:")
    print(null_summary_final_filtered)
else:
    print("Все пропуски обработаны")

In [ ]:
# -----------------------------------------------------------------------------
# 7.11. Проверка распределения целевой переменной
# -----------------------------------------------------------------------------
print("=" * 70)
print("РАСПРЕДЕЛЕНИЕ ЦЕЛЕВОЙ ПЕРЕМЕННОЙ (SALARY_MEDIAN)")
print("=" * 70)
print(feature_df['salary_median'].describe())

# Проверка на выбросы после очистки
print("Проверка на экстремальные значения:")
print(f"Минимум: {feature_df['salary_median'].min():,.0f}")
print(f"Максимум: {feature_df['salary_median'].max():,.0f}")
print(f"Медиана: {feature_df['salary_median'].median():,.0f}")
print(f"Среднее: {feature_df['salary_median'].mean():,.0f}")

In [ ]:
feature_df.write_parquet(DATA_ROOT + "processed\\features.parquet")